In [1]:
import pandas as pd
data=pd.read_csv(r'C:\Users\nguye\Downloads\aclImdb_v1\imdb_dataset.csv')
data.head()

,text,label
0,Bromwell High is a cartoon comedy. It ran at t...,1
1,Homelessness (or Houselessness as George Carli...,1
2,Brilliant over-acting by Lesley Ann Warren. Be...,1
3,This is easily the most underrated film inn th...,1
4,This is not the typical Mel Brooks film. It wa...,1


In [2]:
# lọc ra 1 vài đòn
print(data[1:3])

                                                text  label
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1


In [3]:
data['text'].unique()

array(['Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High\'s satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers\' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I\'m here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn\'t!',
       'Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were once considered human who did ev

In [4]:
data['label'].value_counts()

label
1    25000
0    25000
Name: count, dtype: int64

In [5]:
import re
import html
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# tải dữ liệu NLP cần thiết 
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nguye\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nguye\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nguye\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nguye\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [6]:
def clean_text_svm(text):
    """
    Làm sạch text cho mô hình truyền thống như SVM, Logistic Regression.
    - Xóa HTML, ký tự đặc biệt, stopwords
    - Lemmatize để chuẩn hóa từ
    - Giữ lại chỉ các từ có ý nghĩa
    """
    # 1️ Giải mã ký tự HTML (ví dụ &amp; → &)
    text = html.unescape(text)

    # 2️ Loại bỏ thẻ HTML (<br />, <p>, ...)
    text = re.sub(r'<.*?>', ' ', text)

    # 3️ Đưa về chữ thường
    text = text.lower()

    # 4️ Chuẩn hóa Unicode (PokĂ©mon → Pokemon)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")

    # 5️ Bỏ ký tự không phải chữ cái
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 6️ Thu gọn khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    # 7️ Tokenization
    tokens = word_tokenize(text)

    # 8️ Bỏ stopwords
    tokens = [w for w in tokens if w not in STOPWORDS]

    # 9️ Lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    #  Ghép lại
    return " ".join(tokens)

In [7]:
svm_data = data['text'].apply(clean_text_svm)

In [8]:
print(svm_data.head(10))

0    bromwell high cartoon comedy ran time program ...
1    homelessness houselessness george carlin state...
2    brilliant acting lesley ann warren best dramat...
3    easily underrated film inn brook cannon sure f...
4    typical mel brook film much less slapstick mov...
5    comedic robin williams quirky insane robin wil...
6    yes art successfully make slow paced thriller ...
7    critically acclaimed psychological thriller ba...
8    night listener robin williams toni collette bo...
9    know robin williams god bless constantly shoot...
Name: text, dtype: object


In [9]:
print(svm_data[0])

bromwell high cartoon comedy ran time program school life teacher year teaching profession lead believe bromwell high satire much closer reality teacher scramble survive financially insightful student see right pathetic teacher pomp pettiness whole situation remind school knew student saw episode student repeatedly tried burn school immediately recalled high classic line inspector sack one teacher student welcome bromwell high expect many adult age think bromwell high far fetched pity


In [10]:
svm_df = pd.DataFrame({
    'text': svm_data,
    'label': data['label']
})

In [11]:
svm_df.head()

,text,label
0,bromwell high cartoon comedy ran time program ...,1
1,homelessness houselessness george carlin state...,1
2,brilliant acting lesley ann warren best dramat...,1
3,easily underrated film inn brook cannon sure f...,1
4,typical mel brook film much less slapstick mov...,1


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
import numpy as np
import time
import os
from sklearn.metrics import  (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
import numpy as np

In [19]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)


In [20]:

X = tfidf.fit_transform(svm_df['text'])
y = svm_df['label']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []
f1_scores = []
precisions = []
recalls = []
train_times = []
fold = 1

for train_index, val_index in skf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_val = X[train_index], X[val_index]
    y_train = y.iloc[train_index] if hasattr(y, "iloc") else y[train_index]
    y_val   = y.iloc[val_index]   if hasattr(y, "iloc") else y[val_index]

    model = LinearSVC(C=1.0, random_state=42)

    # Train time
    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()
    train_times.append(end - start)
    print(f"Train time: {end - start:.4f} seconds")

    # Predict
    y_pred = model.predict(X_val)

    # Print sample predictions
    print("----- 5 samples in this fold -----")
    num_samples = min(5, len(val_index))

    for i in range(num_samples):
        real_idx = val_index[i]
        print("-----")
        print(f"Text: {svm_df['text'].iloc[real_idx]}")
        print(f"Label thật: {svm_df['label'].iloc[real_idx]}")
        print(f"Dự đoán: {y_pred[i]}")

    # Metrics
    acc = accuracy_score(y_val, y_pred)
    accuracies.append(acc)

    f1 = f1_score(y_val, y_pred, average='macro')
    f1_scores.append(f1)

    prec = precision_score(y_val, y_pred, average='macro')
    precisions.append(prec)

    rec = recall_score(y_val, y_pred, average='macro')
    recalls.append(rec)

    print(f"Accuracy: {acc:.4f}")
    print(f"F1-macro: {f1:.4f}")
    print(f"Precision-macro: {prec:.4f}")
    print(f"Recall-macro: {rec:.4f}")

    # ===== CONFUSION MATRIX for this fold =====
    cm = confusion_matrix(y_val, y_pred)
    print("\nConfusion Matrix for this fold:")
    print(cm)

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred, digits=4))

    fold += 1


# ===== FINAL SUMMARY =====
print("\n==============================")
print(f"Average Accuracy 5-fold: {np.mean(accuracies):.4f}")
print(f"Accuracy STD: {np.std(accuracies):.4f}")
print("------------------------------")
print(f"Average F1-macro: {np.mean(f1_scores):.4f}")
print(f"F1 STD: {np.std(f1_scores):.4f}")
print("------------------------------")
print(f"Average Precision: {np.mean(precisions):.4f}")
print(f"Average Recall: {np.mean(recalls):.4f}")
print("------------------------------")
print(f"Average train time: {np.mean(train_times):.4f} seconds")
print("==============================")



===== Fold 1 =====
Train time: 1.4820 seconds
----- 5 samples in this fold -----
-----
Text: night listener robin williams toni collette bobby cannavale rory culkin joe morton sandra oh john cullum lisa emery becky ann baker dir patrick stettner hitchcockian suspenser give williams stand low key performance celebrity fan near paranoia one associate almost norm latest derange fan scenario based true event less williams star talk radio personality named gabriel one read story penned airwave accumulated interesting fan form young boy named pete logand culkin submitted manuscript travail troubled youth one editor ashe morton give one read one naturally disturbed ultimately intrigued nightmarish existence pete abducted sexually abused year finally rescued nurse named donna collette giving excellent performance adopted boy correspondence one reveals pete dying aid naturally one want meet fan suddenly doubt possibly devious ulterior motif seed planted estranged lover jess cannavale whose sud

In [21]:
from sklearn.svm import LinearSVC
import joblib

# Train lại toàn bộ data
final_model = LinearSVC(C=1.0, random_state=42)
final_model.fit(X, y)


LinearSVC(random_state=42)

In [22]:
# Lưu model
joblib.dump(final_model, "svm_model.pkl")

# Lưu TF-IDF vectorizer
joblib.dump(tfidf, "tfidf.pkl")

['tfidf.pkl']